<a href="https://colab.research.google.com/github/RubaEgbaria/RAG/blob/master/answers_with_refs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal of this notebook is to find the reference link and add it to the answer to ensure the answers are provided with references.

In [7]:
import pandas as pd

data = pd.read_csv("train_qa.csv")
data.head()

,instruction,question,context,context_text,answer
0,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو برز من عائلة الأحمد؟,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...
1,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,أين وُلد برز من عائلة الأحمد؟,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,ولد في قرية الشجرة في الربع الأخير من القرن ال...
2,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو حسن إبراهيم الباير؟,حسن إبراهيم الباير ولد حسن الباير في قرية برقي...,حسن إبراهيم الباير ولد حسن الباير في قرية برقي...,حسن إبراهيم الباير ولد حسن الباير في قرية برقي...
3,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,ما الذي حدث في استشهاد محمد عبد الرحمن قدورة ع...,اعدام على مرأى من الوالد والأهالي بلغت حصيلة م...,اعدام على مرأى من الوالد والأهالي بلغت حصيلة م...,كما قال أحد الرواة: اعدام على مرأى من الوالد و...
4,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هي وردة حبيب الله أمارة؟,الحاجة وردة حبيب الله أمارة (أم فخري) من كفركن...,الحاجة وردة حبيب الله أمارة (أم فخري) من كفركن...,الحاجة وردة حبيب الله أمارة (أم فخري) من كفركن...


In [4]:
# the current instruction to all records ...
data['instruction'][0]

'أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، متخصص في روايات القرى والمدن الفلسطينيه. أجب بناءً على الشهادات الشفهية والمصادر المحلية الموثّقة، مع الحفاظ على أصالة السياق الإنساني.'

In [8]:
docs = pd.read_csv("documents.csv")
docs.head()

,id,ref,ref_type,title,text,word_count,published_date,scraped_at,transcript_source,part,duration_min,speaker,year_of_birth,gender,chunk_index
0,web_0c9c3c5b,https://www.howiyya.com/VwFamiliesStoriesView?...,article,الشاعر علي الأحمد,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,51,06/03/2009,2026-05-08 12:59:11,NaN,0,0,unknown,NaN,unknown,0
1,web_3acf0191,https://www.howiyya.com/VwFamiliesStoriesView?...,article,حسن إبراهيم الباير,حسن إبراهيم الباير ولد حسن الباير في قرية برقي...,119,16/04/2009,2026-05-08 12:59:12,NaN,0,0,unknown,NaN,unknown,0
2,web_ba3b82b4,https://www.howiyya.com/VwFamiliesStoriesView?...,article,استشهاد محمد عبد الرحمن قدورة على مرأى من والد...,اعدام على مرأى من الوالد والأهالي بلغت حصيلة م...,63,27/04/2009,2026-05-08 12:59:14,NaN,0,0,unknown,NaN,unknown,0
3,web_8cdfdca2,https://www.howiyya.com/VwFamiliesStoriesView?...,article,الفلسطينية أم فخري.. ماتت مرتين بفارق 51 عاما,الحاجة وردة حبيب الله أمارة (أم فخري) من كفركن...,448,02/05/2009,2026-05-08 12:59:16,NaN,0,0,unknown,NaN,unknown,0
4,web_4a0df8ce,https://www.howiyya.com/VwFamiliesStoriesView?...,article,طحيبش والحيش,عائلة طحيبش من بلدة عمقا، وكثير من أبنائها يعي...,23,09/05/2009,2026-05-08 12:59:17,NaN,0,0,unknown,NaN,unknown,0


In [37]:
import re
import pandas as pd

def normalizeText(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\ufeff", "")
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.replace("ـ", "")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text
data["match_key"] = data["context_text"].apply(normalizeText).str[:10]
docs["match_key"] = docs["text"].apply(normalizeText).str[:10]

docsLookup = (
    docs[["match_key", "ref"]]
    .drop_duplicates(subset=["match_key"])
    .rename(columns={"ref": "ref_link"})
)

data = data.drop(columns=["ref_link"], errors="ignore")

data = data.merge(
    docsLookup,
    on="match_key",
    how="left"
)

data = data.drop(columns=["match_key"])

data[["context_text", "ref_link"]].head()

,context_text,ref_link
0,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,https://www.howiyya.com/VwFamiliesStoriesView?...
1,برز من عائلة الأحمد في بلدة الشجرة، علي الأحمد...,https://www.howiyya.com/VwFamiliesStoriesView?...
2,حسن إبراهيم الباير ولد حسن الباير في قرية برقي...,https://www.howiyya.com/VwFamiliesStoriesView?...
3,اعدام على مرأى من الوالد والأهالي بلغت حصيلة م...,https://www.howiyya.com/VwFamiliesStoriesView?...
4,الحاجة وردة حبيب الله أمارة (أم فخري) من كفركن...,https://www.howiyya.com/VwFamiliesStoriesView?...


In [38]:
data["ref_link"].isna().sum()

np.int64(69)

In [39]:
data = data.drop(columns=["ref_link_x", "ref_link_y", "ref_x", "ref_y", "ref"], errors="ignore")

In [42]:
data[50:60]

,instruction,question,context,context_text,answer,ref_link
50,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو حمدي سعيد أمدوخ ،له؟,"حمدي سعيد أمدوخ ،له كتاب ""المختصر المفيد في مع...","حمدي سعيد أمدوخ ،له كتاب ""المختصر المفيد في مع...","حمدي سعيد أمدوخ ،له كتاب ""المختصر المفيد في مع...",https://www.howiyya.com/VwFamiliesStoriesView?...
51,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو الأموي في صفد ،ولد؟,الأموي في صفد ،ولد سنة 1918م،وتعلم في صفد,الأموي في صفد ،ولد سنة 1918م،وتعلم في صفد التح...,الأموي في صفد ،ولد سنة 1918م،وتعلم في صفد,https://www.howiyya.com/VwFamiliesStoriesView?...
52,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو جورج أنطونيوس القدس ،ولد؟,جورج أنطونيوس (القدس) ،ولد في الاسكندرية سنة 1...,جورج أنطونيوس (القدس) ،ولد في الاسكندرية سنة 1...,جورج أنطونيوس (القدس) ،ولد في الاسكندرية سنة 1...,https://www.howiyya.com/VwFamiliesStoriesView?...
53,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو إيراني في عكا،مؤيد إبراهيم؟,إيراني في عكا،مؤيد إبراهيم إيراني،ولد في عكا ع...,إيراني في عكا،مؤيد إبراهيم إيراني،ولد في عكا ع...,إيراني في عكا،مؤيد إبراهيم إيراني،ولد في عكا ع...,https://www.howiyya.com/VwFamiliesStoriesView?...
54,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو الإيراني في يافا،محمود سيف؟,الإيراني في يافا،محمود سيف الدين الإيراني،ولد ...,الإيراني في يافا،محمود سيف الدين الإيراني،ولد ...,الإيراني في يافا،محمود سيف الدين الإيراني،ولد ...,https://www.howiyya.com/VwFamiliesStoriesView?...
55,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو عبد الرحمن بارود أديب؟,الدكتور عبد الرحمن بارود:أديب شاعر مبدع ،وأستا...,الدكتور عبد الرحمن بارود:أديب شاعر مبدع ،وأستا...,الدكتور عبد الرحمن بارود:أديب شاعر مبدع ،وأستا...,https://www.howiyya.com/VwFamiliesStoriesView?...
56,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو حسين محمد البالي م؟,أديب شاعر فقيه له مصنفات,حسين محمد البالي (1820-1855م) أديب شاعر فقيه ل...,أديب شاعر فقيه له مصنفات,https://www.howiyya.com/VwFamiliesStoriesView?...
57,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو البحاروة ،أو دار أبو؟,البحاروة ،أو دار (أبو بحرية)فخذ من آل شراب في ...,البحاروة ،أو دار (أبو بحرية)فخذ من آل شراب في ...,البحاروة ،أو دار (أبو بحرية)فخذ من آل شراب في ...,https://www.howiyya.com/VwFamiliesStoriesView?...
58,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو جميل البحري؟,جميل البحري من حيفا ،كاتب قصة ومسرحية ،وصحفي و...,جميل البحري من حيفا ،كاتب قصة ومسرحية ،وصحفي و...,جميل البحري من حيفا ،كاتب قصة ومسرحية ،وصحفي و...,https://www.howiyya.com/VwFamiliesStoriesView?...
59,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,من هو حسن بن حسن البحيري؟,حسن بن حسن البحيري ولد سنة 1921م في حيفا، من ف...,حسن بن حسن البحيري ولد سنة 1921م في حيفا، من ف...,حسن بن حسن البحيري ولد سنة 1921م في حيفا، من ف...,https://www.howiyya.com/VwFamiliesStoriesView?...


In [43]:
# make the generated answer include the ref_link as a reference.
data["answer"] = data.apply(
    lambda row: f'{row["answer"]}\n\nهذا الجواب مستند إلى المصدر التالي: {row["ref_link"]}'
    if pd.notna(row["ref_link"])
    else row["answer"],
    axis=1
)

In [46]:
data.tail()

,instruction,question,context,context_text,answer,ref_link
17441,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,كيف كانت أجواء رمضان في القرى الفلسطينية قبل ا...,فلسطين - التهجير والنكبة,كانت أجواء رمضان مليئة بالتعاون والبركة، حيث ا...,كانت أجواء رمضان مليئة بالتعاون والبركة، حيث ا...,NaN
17442,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,كيف كان الناس يعرفون موعد بداية رمضان والعيد؟,رمضان - المناسبات والطقوس,كان أهل القرية يعتمدون على الراديو الوحيد المو...,كان أهل القرية يعتمدون على الراديو الوحيد المو...,https://www.howiyya.com/VwFamiliesStoriesView?...
17443,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,ما دور المسحراتي في القرى الفلسطينية؟,فلسطين - التهجير والنكبة,كان المسحراتي يوقظ الناس للسحور ويُعتبر جزءاً ...,كان المسحراتي يوقظ الناس للسحور ويُعتبر جزءاً ...,NaN
17444,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,كيف كان الناس يستعدون لشهر رمضان في فلسطين قبل...,فلسطين - التهجير والنكبة,قبل عام 1948، كان الناس يعتمدون على ما تزرعه أ...,قبل عام 1948، كان الناس يعتمدون على ما تزرعه أ...,NaN
17445,أنت راوٍ وحارس للذاكرة الجماعية الفلسطينية، مت...,كيف كانت رده فعل الفلسطينين جراء مرور ذكرى الن...,اللجون - التهجير والنكبة,حسب المصادر حدث في ال2012: عم الإضراب العام ال...,حسب المصادر حدث في ال2012: عم الإضراب العام ال...,NaN


In [47]:
from google.colab import files

data.to_csv("data_qa.csv", index=False, encoding="utf-8-sig")
files.download("data_qa.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>